# Tech Challenge 2 - Interpretacao de Resultados com LLM

Este notebook atende ao bloco de integracao com LLMs: utiliza Groq (Llama) para transformar a classificacao do modelo em uma explicacao textual destinada a revisao medica.

A LLM nao realiza novo diagnostico. Ela recebe a probabilidade prevista e as cinco principais contribuicoes locais derivadas do modelo servido, sem identificadores do paciente, e explica esses sinais com restricoes de seguranca.

## Estrategia de prompt engineering

O prompt separa instrucoes fixas e contexto numerico. As instrucoes obrigam:

- linguagem em portugues orientada a profissional de saude;
- uso do termo `resultado do modelo`, sem afirmar diagnostico confirmado;
- quatro secoes padronizadas: resumo, evidencias, revisao clinica e limitacoes;
- proibicao de prescricao terapeutica;
- declaracao da natureza academica do modelo e da ausencia de validacao externa.

O modelo padrao e `llama-3.1-8b-instant`, configuravel pela variavel `LLM_MODEL`.

In [1]:
from pathlib import Path
import os
import sys

import joblib
import pandas as pd

ROOT = Path.cwd().resolve()
if ROOT.name == 'notebooks':
    ROOT = ROOT.parent
sys.path.insert(0, str(ROOT))

from src.evaluate_llm import evaluate_cases, predict_case, select_representative_cases
from src.llm_interpretation import (
    DEFAULT_LLM_MODEL,
    SYSTEM_INSTRUCTIONS,
    build_interpretation_prompt,
    derive_feature_evidence,
)

MODEL_PATH = ROOT / 'resultados' / 'fase2' / 'modelo_serving.joblib'
DATA_PATH = ROOT / 'data' / 'cancer_mama.csv'
OUTPUT_DIR = ROOT / 'resultados' / 'fase2'
artifact = joblib.load(MODEL_PATH)

_llm = os.getenv('LLM_MODEL', DEFAULT_LLM_MODEL)
print(f"Modelo interpretado: {artifact['model_label']}")
print(f"LLM configurada: {_llm}")
print(f"GROQ_API_KEY configurada: {bool(os.getenv('GROQ_API_KEY'))}")

Modelo interpretado: Regressao Logistica
LLM configurada: llama-3.1-8b-instant
GROQ_API_KEY configurada: False


## Casos representativos para avaliacao

A avaliacao seleciona tres perfis: maior probabilidade maligna, maior probabilidade benigna e caso mais proximo do limiar de decisao. Assim, o texto e testado em cenarios de alto risco, baixo risco e incerteza relativa.

In [2]:
casos = select_representative_cases(artifact, DATA_PATH)
resumo_casos = []
for nome, row in casos:
    resultado = predict_case(artifact, row)
    resumo_casos.append({
        'caso': nome,
        'classificacao_estimada': resultado.diagnosis,
        'probabilidade_maligna': resultado.probability_malignant,
        'probabilidade_benigna': resultado.probability_benign,
    })
pd.DataFrame(resumo_casos).style.format({
    'probabilidade_maligna': '{:.2%}',
    'probabilidade_benigna': '{:.2%}',
})

,caso,classificacao_estimada,probabilidade_maligna,probabilidade_benigna
0,alto_risco_maligno,Maligno,100.00%,0.00%
1,alto_risco_benigno,Benigno,0.00%,100.00%
2,caso_limiar,Maligno,54.60%,45.40%


## Contexto enviado a LLM

Abaixo esta o contexto do caso de maior probabilidade maligna. Somente dados derivados necessarios a explicacao sao enviados a LLM.

In [3]:
nome_caso, row = casos[0]
resultado = predict_case(artifact, row)
evidencias = derive_feature_evidence(artifact, row.to_dict())
display(pd.DataFrame([item.__dict__ for item in evidencias]))
print(build_interpretation_prompt(resultado, evidencias))

,feature,value,contribution,direction
0,radius_se,2.8730,-11.483334,Maligno
1,area_se,525.6000,-10.664452,Maligno
2,perimeter_se,21.9800,-6.357465,Maligno
3,area_worst,2499.0000,-2.829874,Maligno
4,concave points_mean,0.1595,-2.746355,Maligno


Contexto do resultado a interpretar:
- Modelo: Regressao Logistica
- Classificacao estimada: Maligno (0)
- Probabilidade estimada de malignidade: 100.00%
- Probabilidade estimada de benignidade: 0.00%

Principais evidencias numericas derivadas do modelo:
- radius_se: valor=2.873; contribuicao_local=-11.4833; direcao=Maligno
- area_se: valor=525.6; contribuicao_local=-10.6645; direcao=Maligno
- perimeter_se: valor=21.98; contribuicao_local=-6.3575; direcao=Maligno
- area_worst: valor=2499; contribuicao_local=-2.8299; direcao=Maligno
- concave points_mean: valor=0.1595; contribuicao_local=-2.7464; direcao=Maligno

Produza uma explicacao concisa para apoiar revisao medica, observando integralmente as regras.



## Avaliacao da qualidade das interpretacoes

Quando `GROQ_API_KEY` estiver configurada, a proxima celula gera explicacoes reais para os tres casos e salva:

- `resultados/fase2/interpretacoes_llm.json`, com os textos e metadados;
- `resultados/fase2/avaliacao_interpretacoes_llm.csv`, com uma rubrica objetiva.

A rubrica verifica: mencao da classe prevista, probabilidade, secoes obrigatorias, limitacao explicita, orientacao de revisao profissional e ausencia de prescricao. A avaliacao automatica deve ser complementada por revisao humana de um profissional de saude.

In [4]:
os.environ.setdefault("GROQ_API_KEY", "")  # cole sua chave de console.groq.com/keys

if os.getenv('GROQ_API_KEY'):
    avaliacao = evaluate_cases(MODEL_PATH, DATA_PATH, OUTPUT_DIR)
    display(avaliacao.style.format({'probabilidade_maligna': '{:.2%}', 'score_objetivo': '{:.2%}'}))
else:
    print('GROQ_API_KEY nao configurada.')
    print('Obtenha sua chave gratuita em console.groq.com/keys')
    print('e cole no campo acima antes de reexecutar.')

,caso,classificacao,probabilidade_maligna,menciona_resultado_esperado,inclui_probabilidade,inclui_secoes_obrigatorias,declara_limitacao,orienta_revisao_profissional,nao_prescreve_tratamento,score_objetivo
0,alto_risco_maligno,Maligno,100.00%,True,True,True,False,True,True,83.33%
1,alto_risco_benigno,Benigno,0.00%,True,True,True,False,True,True,83.33%
2,caso_limiar,Maligno,54.60%,True,True,True,False,True,True,83.33%


## Uso pela API

Com `GROQ_API_KEY` definida e `python src/api.py` em execucao, o endpoint `POST /interpret` recebe as mesmas 30 features do `/predict` e retorna classificacao, probabilidades, evidencias locais, explicacao da LLM, versao do prompt e verificacoes da rubrica.

O endpoint `/metrics` registra quantidade de interpretacoes, latencia e distribuicao da nota objetiva, permitindo acompanhar a camada LLM em operacao.